# Audio splitter — chapter 1 (final stage)

Take a canonical *raw* JSONL produced by some `prep_*.ipynb` and **cut the actual per-instance WAV files** referenced by every `audio_path`.

**Input**
- `data/processed_jsonl/<dataset>_instance.raw.jsonl` — every record's `audio_path` is a *predicted* future location (the file doesn't exist yet).
- `data/unpacked/<dataset>/...` — the source WAVs the prep notebook referred to via `metadata.source_file`.

**Output**
- `data/cut_audio/<dataset>/...` — one 16 kHz mono PCM-16 WAV per record.
- `data/processed_jsonl/<dataset>_instance.jsonl` — same content as the raw JSONL but with `audio_path` now pointing at files that exist.

**What this notebook does NOT do**
- Edit labels, splits, or text — that's the prep notebook's job.
- Apply VAD, denoising, or any other DSP beyond resample-to-16k + trim.
- Keep records whose source WAV can't be resolved. They get dropped. (No half-baked records in the output.)

**Convention.** Every function in this notebook will eventually become a function in `audio_splitter.py`. For now, notebook-first.

---

## 0. Setup

The only import we need from this project is `utils_dataprep` (canonical JSONL I/O, schema validation, path helpers). Everything else is in the cells below.

In [1]:
import sys
from pathlib import Path

# Make sure we can import utils_dataprep no matter where Jupyter was launched.
HERE = Path.cwd()
if HERE.name != "1_data_prep":
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp

PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Chapter dir = {HERE}")

PROJECT_ROOT = /home/ivan/Posao_IJS/Stepping_Stones/github_full_repo/slavic-speech-pipeline
Chapter dir = /home/ivan/Posao_IJS/Stepping_Stones/github_full_repo/slavic-speech-pipeline/1_data_prep


Standard third-party imports — already in `requirements.txt` / `setup_env.sh`.

In [2]:
from collections import OrderedDict
from dataclasses import dataclass

import numpy as np
import soundfile as sf
from tqdm.auto import tqdm

---

## 1. Config

All knobs are here. Edit, re-run the cells below.

**Conventions**
- All paths are project-relative strings.
- `source_audio_root` is recursively scanned for `*.wav` files. Lookup happens by stem (e.g. `metadata.source_file="ROG-Dia-GSO-P0005.exb"` → stem `ROG-Dia-GSO-P0005` → match the WAV with that stem under the root).
- `force=True` re-cuts existing destinations. Leave `False` for cheap re-runs.
- `test_mode=True` processes the first `test_n` records and writes outputs to `data/test_processed_jsonl/` so production files are safe.

In [3]:
@dataclass
class Config:
    # Input (a *.raw.jsonl produced by some prep_*.ipynb)
    input_jsonl: str = "data/processed_jsonl/rog_dialog_instance.raw.jsonl"
    # Output (canonical JSONL with audio_path pointing at real cut WAVs)
    output_jsonl: str = "data/processed_jsonl/rog_dialog_instance.jsonl"
    # Recursive root to scan for source WAVs. Anything below this with a .wav
    # extension is added to the index; lookups are by stem.
    source_audio_root: str = "data/unpacked/ROG-Dialog"

    # Re-cut destinations even if they already exist.
    force: bool = False
    # Warn (but keep) cuts shorter than this many ms. Set to 0 to silence.
    short_cut_warn_ms: int = 100
    # Cache up to this many source WAVs in memory. ROG-Dialog has hundreds of
    # segments per source, so even 1 is huge; 4 gives breathing room when
    # records aren't perfectly clustered by file_id.
    source_cache_size: int = 4

    # Test mode
    test_mode: bool = False                                                     ################ TEST MODE
    test_n: int = 10


cfg = Config()
print(cfg)
if cfg.test_mode:
    udp.banner("🧪 TEST MODE ENABLED", char="-")

Config(input_jsonl='data/processed_jsonl/rog_dialog_instance.raw.jsonl', output_jsonl='data/processed_jsonl/rog_dialog_instance.jsonl', source_audio_root='data/unpacked/ROG-Dialog', force=False, short_cut_warn_ms=100, source_cache_size=4, test_mode=False, test_n=10)


---

## 2. Source-WAV resolution

We walk `source_audio_root` once, indexing every `.wav` by stem. Records are then resolved via `metadata.source_file` (set by the prep notebook), stripping its extension.

Collisions — two WAVs with the same stem under different subfolders — are reported. First one wins; that's almost certainly a dataset-layout bug worth flagging.

In [4]:
def build_source_index(source_audio_root: str) -> dict[str, Path]:
    """Walk `source_audio_root` recursively, return {stem: absolute_wav_path}."""
    root = udp.from_project_relative(source_audio_root)
    if not root.exists():
        raise FileNotFoundError(
            f"source_audio_root does not exist: {root}\n"
            f"Set Config.source_audio_root."
        )

    index: dict[str, Path] = {}
    collisions: list[tuple[str, Path, Path]] = []
    for wav in root.rglob("*.wav"):
        stem = wav.stem
        if stem in index:
            collisions.append((stem, index[stem], wav))
            continue
        index[stem] = wav

    if collisions:
        print(f"⚠️  {len(collisions)} duplicate WAV stems under {root}; keeping the first:")
        for stem, kept, dropped in collisions[:5]:
            print(f"   - {stem}: kept {kept.relative_to(root)}, ignored {dropped.relative_to(root)}")
        if len(collisions) > 5:
            print(f"   ... and {len(collisions) - 5} more")

    return index


def resolve_source(record: dict, index: dict[str, Path]) -> Path | None:
    """
    Look up the source WAV for `record` in `index`. Returns absolute Path or None.

    Strategy: take `metadata.source_file` (the authoritative field set by the
    prep notebook), strip its extension, and look up by stem.
    """
    src_file = record.get("metadata", {}).get("source_file")
    if not src_file:
        return None
    stem = Path(src_file).stem  # "foo.exb" -> "foo"
    return index.get(stem)

---

## 3. Audio I/O — the slim inner loop

`SourceCache` is a tiny LRU that holds decoded source arrays. Without it, a 5-minute source WAV would be re-decoded for every one of its hundreds of segments. With it, each source is decoded exactly once.

`cut_slice` is the hot path: take the cached array, slice in samples, resample to 16 kHz, write PCM-16.

`cut_whole_file` handles the (chapter-4) case where there's no `start_t`/`end_t` — delegate to `udp.resample_to_16k_mono`.

In [5]:
class SourceCache:
    """Tiny LRU cache for decoded source WAVs (data, sr) keyed by absolute path."""

    def __init__(self, max_size: int = 4) -> None:
        self.max_size = max(1, int(max_size))
        self._cache: OrderedDict[str, tuple[np.ndarray, int]] = OrderedDict()

    def get(self, path: Path) -> tuple[np.ndarray, int]:
        key = str(path)
        if key in self._cache:
            self._cache.move_to_end(key)
            return self._cache[key]
        data, sr = sf.read(key, dtype="float32", always_2d=False)
        if data.ndim == 2:
            data = data.mean(axis=1)
        self._cache[key] = (data, sr)
        if len(self._cache) > self.max_size:
            self._cache.popitem(last=False)
        return data, sr


def _resample_array_to_16k(data: np.ndarray, sr: int) -> np.ndarray:
    """Resample a 1-D float32 array to 16 kHz. Prefer librosa; fall back to scipy."""
    if sr == 16000:
        return data
    try:
        import librosa
        return librosa.resample(data, orig_sr=sr, target_sr=16000)
    except ImportError:
        from math import gcd
        from scipy.signal import resample_poly
        g = gcd(sr, 16000)
        return resample_poly(data, 16000 // g, sr // g).astype(np.float32)


def cut_slice(record: dict, cache: SourceCache, src_path: Path, dst_path: Path) -> bool:
    """Slice [start_t, end_t] from cached source, resample to 16 kHz, write 16-bit PCM."""
    try:
        data, sr = cache.get(src_path)
        st = float(record["start_t"])
        et = float(record["end_t"])
        s = max(0, int(st * sr))
        e = min(len(data), int(et * sr))
        if e <= s:
            print(f"⚠️  {record['instance_id']}: end <= start after sample-conversion (s={s}, e={e}); dropping")
            return False
        clip = data[s:e]
        clip = _resample_array_to_16k(clip, sr)
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(str(dst_path), clip, 16000, subtype="PCM_16")
        return True
    except Exception as ex:
        print(f"⚠️  {record['instance_id']}: cut failed ({ex}); dropping")
        return False


def cut_whole_file(record: dict, src_path: Path, dst_path: Path) -> bool:
    """Whole-file copy + resample for records without start_t/end_t."""
    try:
        udp.resample_to_16k_mono(src_path, dst_path)
        return True
    except Exception as ex:
        print(f"⚠️  {record['instance_id']}: whole-file copy failed ({ex}); dropping")
        return False

---

## 4. Main loop

`process_records` orchestrates everything:
1. Sort records by source path so cache hits hard (all segments of one source processed back-to-back).
2. For each record: resolve source → check destination exists → cut → record outcome.
3. Records that can't be cut are dropped from `kept`. Reasons are counted in `stats`.

In [6]:
def process_records(records: list[dict], index: dict[str, Path], cfg: Config) -> tuple[list[dict], dict]:
    """Cut every record. Returns (kept_records, stats_dict)."""
    cache = SourceCache(max_size=cfg.source_cache_size)
    stats = {"input": len(records), "missing_source": 0, "cut_failed": 0,
             "skipped_existing": 0, "kept": 0, "short_warned": 0}

    # Sort by source path for cache locality. Records with no resolvable source
    # land at the front (sort key "") and get dropped immediately.
    def _src_key(r: dict) -> str:
        p = resolve_source(r, index)
        return str(p) if p else ""
    records = sorted(records, key=_src_key)

    pbar = tqdm(records, desc="Cutting", unit="rec")
    kept: list[dict] = []
    for rec in pbar:
        src = resolve_source(rec, index)
        if src is None:
            stats["missing_source"] += 1
            continue

        dst_abs = udp.from_project_relative(rec["audio_path"])

        # Idempotency
        if dst_abs.exists() and dst_abs.stat().st_size > 0 and not cfg.force:
            stats["skipped_existing"] += 1
            rec["audio_path"] = udp.to_project_relative(dst_abs)
            kept.append(rec)
            continue

        has_slice = "start_t" in rec and "end_t" in rec
        ok = cut_slice(rec, cache, src, dst_abs) if has_slice else cut_whole_file(rec, src, dst_abs)

        if not ok:
            stats["cut_failed"] += 1
            continue

        # Short-cut warning (after success)
        if has_slice and cfg.short_cut_warn_ms > 0:
            duration_ms = (float(rec["end_t"]) - float(rec["start_t"])) * 1000.0
            if duration_ms < cfg.short_cut_warn_ms:
                stats["short_warned"] += 1
                pbar.write(f"   short cut ({duration_ms:.1f} ms): {rec['instance_id']}")

        rec["audio_path"] = udp.to_project_relative(dst_abs)
        kept.append(rec)
        stats["kept"] += 1

    return kept, stats

---

## 5. Load and validate the input JSONL

Every line must already conform to the canonical schema. If anything is off, we want to know now — not three hours into a cutting run.

In [7]:
records = udp.read_jsonl(cfg.input_jsonl)
n_total, n_valid, errs = udp.validate_jsonl(records)
print(f"Loaded {n_total} records, {n_valid} valid.")
if errs:
    print("First errors:")
    for e in errs[:5]:
        print(f"   {e}")
    raise ValueError("input JSONL failed canonical-schema validation")

if cfg.test_mode:
    records = records[: cfg.test_n]
    print(f"🧪 capped to {len(records)} records")

# Peek at one
import json as _json
print("\nSample record:")
print(_json.dumps(records[0], ensure_ascii=False, indent=2)[:600])

Loaded 14494 records, 14494 valid.

Sample record:
{
  "instance_id": "ROG-Dialog_ROG-Dia-GSO-P0005_ROG-dialog-0007_0.555_1.133",
  "dataset": "ROG-Dialog",
  "file_id": "ROG-Dia-GSO-P0005",
  "audio_path": "data/cut_audio/ROG-Dialog/ROG-Dia-GSO-P0005_ROG-dialog-0007_0.555_1.133.wav",
  "split": "test",
  "speaker": "ROG-dialog-0007",
  "start_t": 0.555,
  "end_t": 1.133,
  "text": "Kaj zdaj,",
  "labels": {
    "sentiment": "mixedNegative",
    "sentiment_annotated": "mixedNegative",
    "dialogue_act_function": "interactionStructuring",
    "dialogue_act_dimension": "discourseStructuring"
  },
  "metadata": {
    "source_file": "ROG-Dia-GSO-


---

## 6. Build the source-WAV index and check coverage

If coverage is <100%, look at the first few unresolved records and check:
- Is `metadata.source_file` populated correctly by the prep notebook?
- Did `download_data` actually unpack the audio archive (it can fail silently on the GOS restricted handle)?
- Is `source_audio_root` pointed at the right place?

In [8]:
index = build_source_index(cfg.source_audio_root)
print(f"Source index: {len(index)} WAVs under {cfg.source_audio_root}")

resolved = 0
unresolved_samples = []
for r in records:
    if resolve_source(r, index) is not None:
        resolved += 1
    elif len(unresolved_samples) < 5:
        unresolved_samples.append(r)

print(f"Resolved: {resolved}/{len(records)} ({100 * resolved / max(1, len(records)):.1f}%)")
if unresolved_samples:
    print("\nFirst unresolved records (these will be dropped):")
    for r in unresolved_samples:
        print(f"   - {r['instance_id']}  source_file={r['metadata'].get('source_file')!r}")

Source index: 12 WAVs under data/unpacked/ROG-Dialog
Resolved: 14494/14494 (100.0%)


---

## 7. Cut

The hot loop. Sources cached, slices written, failures dropped.

In [9]:
kept, stats = process_records(records, index, cfg)
udp.banner("Stats", char="-")
for k, v in stats.items():
    print(f"  {k:>18}: {v}")

Cutting:   0%|          | 0/14494 [00:00<?, ?rec/s]

   short cut (91.0 ms): ROG-Dialog_ROG-Dia-GSO-P0007_ROG-dialog-0011_1548.958_1549.049
   short cut (67.0 ms): ROG-Dialog_ROG-Dia-GSO-P0007_ROG-dialog-0011_1701.897_1701.964
   short cut (67.0 ms): ROG-Dialog_ROG-Dia-GSO-P0007_ROG-dialog-0012_1701.897_1701.964
   short cut (34.0 ms): ROG-Dialog_ROG-Dia-GSO-P0009_ROG-dialog-0016_1141.730_1141.764
   short cut (86.0 ms): ROG-Dialog_ROG-Dia-GSO-P0011_ROG-dialog-0020_581.324_581.410
   short cut (33.0 ms): ROG-Dialog_ROG-Dia-GSO-P0011_ROG-dialog-0020_625.103_625.136
   short cut (67.0 ms): ROG-Dialog_ROG-Dia-GSO-P0016_ROG-dialog-0029_414.159_414.226
   short cut (98.0 ms): ROG-Dialog_ROG-Dia-GSO-P0018_ROG-dialog-0034_478.984_479.082
   short cut (40.0 ms): ROG-Dialog_ROG-Dia-GSO-P0022_ROG-dialog-0042_544.465_544.505
   short cut (80.0 ms): ROG-Dialog_ROG-Dia-GSO-P0022_ROG-dialog-0042_743.178_743.258

----------------------------------------------------------------------
Stats
----------------------------------------------------------------

---

## 8. Write the output JSONL and re-validate

Test mode mirrors the output path under `data/test_processed_jsonl/` so it can't overwrite real outputs.

In [10]:
out_path = (cfg.output_jsonl
            if not cfg.test_mode
            else cfg.output_jsonl.replace("data/processed_jsonl/", "data/test_processed_jsonl/"))
n_written = udp.write_jsonl(kept, out_path)
print(f"✅ wrote {n_written} records to {out_path}")

n_total, n_valid, errs = udp.validate_jsonl(udp.iter_jsonl(out_path))
if errs:
    print(f"⚠️  output has {n_total - n_valid}/{n_total} invalid records:")
    for e in errs[:5]:
        print(f"   {e}")
else:
    print(f"✅ output validates: {n_valid}/{n_total}")

✅ wrote 14494 records to data/processed_jsonl/rog_dialog_instance.jsonl
✅ output validates: 14494/14494


---

## 9. Spot-check a cut WAV

Open one of the freshly cut files and confirm the duration matches `end_t - start_t` within a few ms, and the sample rate is 16 kHz.

In [11]:
if kept:
    r = kept[0]
    p = udp.from_project_relative(r["audio_path"])
    data, sr = sf.read(str(p))
    dur_actual = len(data) / sr
    if "start_t" in r and "end_t" in r:
        dur_expected = r["end_t"] - r["start_t"]
        print(f"{r['instance_id']}")
        print(f"   sr={sr}, actual={dur_actual:.4f}s, expected={dur_expected:.4f}s, "
              f"delta={1000*(dur_actual - dur_expected):.2f}ms")
    else:
        print(f"{r['instance_id']}: whole-file, sr={sr}, duration={dur_actual:.4f}s")
else:
    print("No records were kept — nothing to spot-check.")

ROG-Dialog_ROG-Dia-GSO-P0005_ROG-dialog-0007_0.555_1.133
   sr=16000, actual=0.5781s, expected=0.5780s, delta=0.06ms


---

## 10. What's next

Chapter 1 is now complete for this dataset. The next chapter is **`2_data_analysis/sniff_dataset.ipynb`** — point it at the JSONL we just wrote (e.g. `data/processed_jsonl/rog_dialog_instance.jsonl`) and look at label distributions, audio-duration histograms, and per-split summaries before touching a model.